# Bar chart generator


In [59]:
from __future__ import annotations

import importlib
import json
import re
import shutil
import subprocess
import sys
import tempfile
from datetime import datetime
from pathlib import Path
from uuid import uuid4

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import colors as mcolors

# Groq
import os
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY_HERE"
from groq import Groq


def _optional_import(module_name):
    try:
        return importlib.import_module(module_name)
    except Exception:
        return None


def install_package(pip_name: str):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])


def ensure_module(module_name: str, pip_name: str | None = None):
    module = _optional_import(module_name)
    if module is not None:
        return module
    install_package(pip_name or module_name)
    return importlib.import_module(module_name)


sns = _optional_import("seaborn")
alt = _optional_import("altair")
go  = _optional_import("plotly.graph_objects")

try:
    from IPython.display import display
except Exception:
    display = None


In [60]:
from pathlib import Path

PROJECT = Path.cwd().resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent

DATA_TRAIN = Path(r"C:\Users\Michelle\I2R\data\train")

# Normal generated outputs
BAR_OUT_ROOT = PROJECT / "outputs" / "generated" / "barplots"

# Testing outputs
BAR_TEST_OUT_ROOT = PROJECT / "testing" / "bar_testing"

BAR_OUT_ROOT.mkdir(parents=True, exist_ok=True)
BAR_TEST_OUT_ROOT.mkdir(parents=True, exist_ok=True)

CLEAR_OUTPUT = True
LIBRARIES = ["altair", "matplotlib", "seaborn", "plotly"]
SUBDIRS = ["images", "tables", "metadata"]

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)


# Data


In [61]:
# ── Dataset registry ──────────────────────────────────────────────────────────
DATASET_REGISTRY = {
    "warehouse_retail": {
        "path":         DATA_TRAIN / "Warehouse_and_Retail_Sales.csv",
        "numeric_cols": ["RETAIL SALES", "WAREHOUSE SALES", "RETAIL TRANSFERS"],
        "group_cols":   ["ITEM TYPE", "SUPPLIER"],
        "date_col":     "date",
        "agg_cols":     ["YEAR", "MONTH"],
        "loader":       "load_warehouse_retail",
    },
    # ── Add new datasets below ────────────────────────────────────────────────
    # "my_dataset": {
    #     "path":         DATA_TRAIN / "my_file.csv",
    #     "numeric_cols": ["col_a"],
    #     "group_cols":   ["category"],
    #     "date_col":     None,
    #     "agg_cols":     None,
    #     "loader":       None,
    # },
}

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)


In [62]:
def load_warehouse_retail(df: pd.DataFrame) -> pd.DataFrame:
    df["date"] = pd.to_datetime(
        dict(year=df["YEAR"].astype("Int64"), month=df["MONTH"].astype("Int64"), day=1),
        errors="coerce",
    )
    return df


def get_loader(name: str):
    loaders = {
        "load_warehouse_retail": load_warehouse_retail,
        # Register new loaders here
    }
    return loaders.get(name)


def load_dataset(dataset_name: str) -> pd.DataFrame | None:
    spec = DATASET_REGISTRY.get(dataset_name)
    if spec is None:
        raise KeyError(f"Unknown dataset: {dataset_name}")
    path = spec["path"]
    if not path.exists():
        print(f"Dataset not found: {path}")
        return None
    df = pd.read_csv(path)
    if spec["loader"]:
        loader_fn = get_loader(spec["loader"])
        if loader_fn:
            df = loader_fn(df)
    return df


DATASETS = {}
for ds_name, ds_spec in DATASET_REGISTRY.items():
    ds = load_dataset(ds_name)
    if ds is not None:
        DATASETS[ds_name] = ds
        print(f"Loaded '{ds_name}': {len(ds)} rows, cols: {list(ds.columns)}")

DEFAULT_DATASET = next(iter(DATASETS)) if DATASETS else None
print(f"Default dataset: {DEFAULT_DATASET}")


Loaded 'warehouse_retail': 307645 rows, cols: ['YEAR', 'MONTH', 'SUPPLIER', 'ITEM CODE', 'ITEM DESCRIPTION', 'ITEM TYPE', 'RETAIL SALES', 'RETAIL TRANSFERS', 'WAREHOUSE SALES', 'date']
Default dataset: warehouse_retail


## 1. Sampling weights

Weights are derived directly from observed frequencies in the reference dataset (n=100).
Each parameter maps `value → count` and is normalised at sample time.


In [63]:
SAMPLING_WEIGHTS = {
    # is_vertical: 0=horizontal bar, 1=vertical bar
    "is_vertical":           {False: 0.26, True: 0.74},

    # title_present: whether a title is shown
    "title_present":         {False: 0.39, True: 0.61},

    # title_location: alignment of title (none only when title_present=False)
    "title_location":        {"none": 0.39, "center": 0.48, "left": 0.13},

    # title_color: black vs dark gray
    "title_color":           {"gray": 0.64, "black": 0.36},

    # title_size: small (~10-12pt) vs average (~12-18pt)
    "title_size":            {"small": 0.51, "average": 0.49},

    # title_size_large: overrides title_size to large (~20-24pt)
    "title_size_large":      {False: 0.88, True: 0.12},

    # subtitle_present
    "subtitle_present":      {False: 0.96, True: 0.04},

    # outline_chart: border around the entire chart area
    "outline_chart":         {False: 0.90, True: 0.10},

    # legend_location: 'no legend' or a position string
    # middle-right(3) → right, top-center(1) → top, bottom-center(1) → bottom
    "legend_location":       {"no legend": 0.91, "right": 0.03, "top": 0.01,
                              "bottom": 0.01, "top-left": 0.01, "top-right": 0.03},

    # legend_title: whether legend has a title label
    "legend_title":          {False: 0.99, True: 0.01},

    # legend_title_color: whether legend title is colored (always False in data)
    "legend_title_color":    {False: 1.00},

    # legend_outline: border around legend box
    "legend_outline":        {False: 0.98, True: 0.02},

    # gridlines: whether any gridlines are shown
    "gridlines":             {False: 0.48, True: 0.52},

    # gridlines_both: horizontal AND vertical gridlines
    "gridlines_both":        {False: 0.87, True: 0.13},

    # x_labels_orientation: visibility of x-axis tick labels
    "x_labels_orientation":  {"hidden": 0.08, "horizontal": 0.92},

    # x_labels_diagonal: x labels at 45 degrees
    "x_labels_diagonal":     {False: 0.96, True: 0.04},

    # x_labels_vertical: x labels at 90 degrees
    "x_labels_vertical":     {False: 0.99, True: 0.01},

    # y_labels_orientation: visibility of y-axis tick labels
    "y_labels_orientation":  {"hidden": 0.02, "horizontal": 0.98},

    # y_labels_diagonal: y labels at 45 degrees (always False in data)
    "y_labels_diagonal":     {False: 1.00},

    # y_labels_vertical: y labels at 90 degrees
    "y_labels_vertical":     {False: 0.99, True: 0.01},

    # labels_colored: axis tick labels rendered in a non-black color
    "labels_colored":        {False: 0.85, True: 0.15},

    # direct_labeling: value labels on bars
    #   none=no labels, direct_no_axis=labels replace axis, direct_with_axis=labels + axis
    "direct_labeling":       {"none": 0.85, "direct_no_axis": 0.02, "direct_with_axis": 0.13},

    # direct_label_position: where the value label sits on the bar
    "direct_label_position": {"base": 0.85, "outside": 0.15},

    # aspect_ratio: overall chart shape
    "aspect_ratio":          {"square": 0.09, "horizontal": 0.89, "vertical": 0.02},

    # sample_bin: approximate number of bars (mapped to a concrete n_bars value)
    "sample_bin":            {8: 0.90, 15: 0.05, 25: 0.02, 35: 0.02, 45: 0.01},

    # colored_bars: multicolor vs single-color bars
    "colored_bars":          {False: 0.03, True: 0.97},

    # palette_type: which colour palette to use when colored_bars=True
    "palette_type":          {4: 0.25, 9: 0.15, 10: 0.15, 11: 0.10,
                              7: 0.08, 14: 0.08, 2: 0.05, 3: 0.05,
                              13: 0.04, 15: 0.03, 16: 0.02},

    # rounded_corners: bars have rounded top corners
    "rounded_corners":       {False: 0.91, True: 0.09},

    # outline_bars: bars have a black border stroke
    "outline_bars":          {False: 0.73, True: 0.27},

    # background_non_white: non-white (light gray) plot background
    # background: white or one of several non-white options
    "background":            {"white": 0.97, "light_gray": 0.01, "light_color": 0.01, "dark_gray": 0.005, "dark": 0.005},

    # ticks: which axes show tick marks
    "ticks":                 {"none": 0.48, "x": 0.08, "both": 0.21, "y": 0.23},
}





In [64]:
TITLE_COLOR_PALETTES = {
    "black": ["#000000", "#1a1a1a","#212121"],
    "gray":  ["#555555", "#6b6b6b", "#888888", "#9e9e9e"],
}

TITLE_SIZE_PALETTES = {
    "small":   [9, 10, 11, 12],
    "average": [13, 14, 15, 16, 17, 18],
    "large":   [20, 21, 22, 23, 24],
}

PALETTES = {
    0:  ["#111111", "#333333", "#555555", "#777777", "#999999"],            # black
    1:  ["#1d3557", "#457b9d", "#264653", "#3a5a40", "#6c757d"],            # dark
    2:  ["#023e8a", "#0077b6", "#0096c7", "#00b4d8", "#48cae4"],            # dark blue
    3:  ["#1b4332", "#2d6a4f", "#40916c", "#52b788", "#74c69d"],            # dark green
    4:  ["#e63946", "#f4a261", "#ffbe0b", "#06d6a0", "#118ab2", "#8338ec"],# colorful
    5:  ["#b7e4c7", "#95d5b2", "#74c69d", "#52b788", "#40916c"],            # light green
    6:  ["#e63946", "#118ab2", "#ef233c", "#4361ee"],                       # red & blue
    7:  ["#a8dadc", "#ffd6a5", "#caffbf", "#fdffb6", "#c8b6ff"],            # light
    8:  ["#e63946", "#2a9d8f", "#264653"],                                  # RGB
    9:  ["#023e8a", "#e76f51", "#219ebc", "#fb8500"],                       # blue & orange
    10: ["#f4a261", "#118ab2", "#e63946", "#2a9d8f"],                       # orange/blue/red/green
    11: ["#118ab2", "#f4a261", "#2a9d8f"],                                  # blue/orange/green
    12: ["#ffb347", "#ffa500", "#ff8c00", "#e07b00"],                       # light orange
    13: ["#e76f51", "#f4a261", "#e9c46a", "#e07b00"],                       # orange
    14: ["#a8dadc", "#457b9d", "#1d3557", "#48cae4"],                       # light blue
    15: ["#f72585", "#b5179e", "#7209b7", "#560bad"],                       # pink shades
    16: ["#e63946", "#c1121f", "#9d0208", "#6a040f"],                       # red
}

BACKGROUND_LIGHT_GRAY  = ["#f0f0f0", "#ebebeb", "#e8e8e8", "#f2f2f2", "#ededed"]
BACKGROUND_DARK_GRAY   = ["#343a40", "#2d3436", "#3d3d3d", "#404040", "#2c2c2c"]
BACKGROUND_LIGHT_COLOR = ["#eef2ff", "#fff3e0", "#e8f5e9", "#fce4ec", "#e3f2fd", "#f3e5f5"]
BACKGROUND_DARK        = ["#1a1a2e", "#16213e", "#1e1e1e", "#212121", "#1a1a1a", "#0d0d0d"]

In [65]:
def normalize_weights(weights: dict) -> dict:
    out = {}
    for param, codes in weights.items():
        total = sum(codes.values())
        out[param] = {k: v / total for k, v in codes.items()}
    return out


def sample_style(rng: np.random.Generator, weights: dict, overrides: dict | None = None) -> dict:
    style = {}
    for param, val_probs in weights.items():
        vals  = list(val_probs.keys())
        probs = np.array(list(val_probs.values()), dtype=float)
        probs /= probs.sum()
        style[param] = vals[int(rng.choice(len(vals), p=probs))]

    # Expand title_color group name → specific hex
    color_group = style.get("title_color")
    if color_group in TITLE_COLOR_PALETTES:
        palette = TITLE_COLOR_PALETTES[color_group]
        style["title_color"] = palette[int(rng.integers(0, len(palette)))]

        

    if overrides:
        style.update(overrides)
    return harmonize_style(style)

## 2. Data sampling


In [66]:
def sample_bar_data_from_df(
    df: pd.DataFrame,
    spec: dict,
    rng: np.random.Generator,
    style: dict,
    min_total: float = 1.0,
) -> tuple[pd.DataFrame, dict] | None:
    numeric_cols = spec["numeric_cols"]
    group_cols   = spec["group_cols"]
    date_col     = spec.get("date_col")

    sample_bin = style.get("sample_bin", 8)
    n_bars = int(sample_bin) if isinstance(sample_bin, int) and sample_bin >= 2 else 8
    n_bars = max(2, min(n_bars, 30))

    value_col    = str(rng.choice(numeric_cols))
    category_col = str(rng.choice(group_cols))

    strategy = str(rng.choice(["group", "time_window", "filtered_group"],
                               p=[0.40, 0.35, 0.25]))
    context = {"value_col": value_col, "strategy": strategy}

    if strategy == "time_window" and date_col and date_col in df.columns:
        window_months = int(rng.choice([1, 3, 6, 12], p=[0.30, 0.35, 0.20, 0.15]))
        monthly_totals = df.groupby(date_col)[value_col].sum()
        valid_months   = monthly_totals[monthly_totals > min_total].index.to_numpy()
        if len(valid_months) == 0:
            return None
        start = pd.to_datetime(rng.choice(valid_months))
        end   = start + pd.DateOffset(months=window_months)
        dfw   = df[(df[date_col] >= start) & (df[date_col] < end)].copy()
        if len(dfw) == 0:
            return None
        series = dfw.groupby(category_col)[value_col].sum()
        context.update({"category_col": category_col, "start": start,
                        "end": end, "window_months": window_months})

    elif strategy == "filtered_group" and len(group_cols) >= 2:
        other_cols = [c for c in group_cols if c != category_col]
        filter_col = str(rng.choice(other_cols))
        candidates = np.array(df[filter_col].dropna().unique(), dtype=str)
        rng.shuffle(candidates)
        dfw = None
        for val in candidates[:10]:
            tmp        = df[df[filter_col] == val].copy()
            tmp_series = tmp.groupby(category_col)[value_col].sum()
            tmp_series = tmp_series[tmp_series > 0]
            if len(tmp_series) >= 2 and tmp_series.sum() > min_total:
                dfw = tmp
                filter_val = str(val)
                break
        if dfw is None:
            return None
        series = dfw.groupby(category_col)[value_col].sum()
        context.update({"category_col": category_col, f"filter_{filter_col}": filter_val})

    else:  # "group"
        series = df.groupby(category_col)[value_col].sum()
        context["category_col"] = category_col

    series = series[series > 0].sort_values(ascending=False)
    if len(series) < 2:
        return None

    series = series.head(n_bars)
    if len(series) < 2:
        return None

    total = float(series.sum())
    if total <= min_total:
        return None

    plot_df = series.reset_index()
    plot_df.columns = [category_col, value_col]

    context.update({
        "category_col": category_col,
        "n_bars":       int(len(plot_df)),
        "total":        total,
    })
    return plot_df, context


## 3. Style helpers


In [67]:
COLOR_PALETTE_HEX = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2",
    "#7f7f7f", "#bcbd22", "#17becf", "#393b79", "#637939", "#8c6d31", "#843c39", "black",
]

SCHEME_NAMES = ["tableau10", "set2", "accent", "category10", "dark2"]
MPL_SCHEMES  = {"tableau10": "tab10", "set2": "Set2", "accent": "Accent",
                "category10": "tab10", "dark2": "Dark2"}

_ALTAIR_VALID_ORIENTS = {
    "none", "left", "right", "top", "bottom",
    "top-left", "top-right", "bottom-left", "bottom-right",
}


def new_chart_id(prefix="bar"):
    stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
    return f"{prefix}_{stamp}_{uuid4().hex[:8]}"


def safe_slug(value):
    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    return value.strip("_") or "value"

DARK_BACKGROUNDS = {"dark_gray", "dark"}

def harmonize_style(style: dict) -> dict:
    style = dict(style)
    if not style.get("title_present", True):
        style["title_location"] = "none"
        style["subtitle_present"] = False
    if style.get("title_size_large"):
        style["title_size"] = "large"
    if style.get("legend_location") == "no legend":
        style["legend_title"]       = False
        style["legend_title_color"] = False
        style["legend_outline"]     = False
    if style.get("gridlines_both"):
        style["gridlines"] = True
    if style.get("x_labels_vertical"):
        style["x_labels_orientation"] = "vertical"
    elif style.get("x_labels_diagonal"):
        style["x_labels_orientation"] = "diagonal"
    if style.get("y_labels_vertical"):
        style["y_labels_orientation"] = "vertical"
    elif style.get("y_labels_diagonal"):
        style["y_labels_orientation"] = "diagonal"
    # Force white text on dark backgrounds
    if style.get("background") in DARK_BACKGROUNDS:
        style["title_color"]    = "#ffffff"
        style["labels_colored"] = False
        style["_force_white_text"] = True
    else:
        style.pop("_force_white_text", None)
    return style


def figure_size(style):
    return {"square": (6, 6), "horizontal": (8, 5), "vertical": (5, 8)}.get(
        style.get("aspect_ratio", "horizontal"), (8, 5))


def axis_angle(orientation, backend="matplotlib"):
    if orientation in {"diagonal"}:
        return -45 if backend == "altair" else 45
    if orientation in {"vertical"}:
        return -90 if backend == "altair" else 90
    return 0


def labels_visible(orientation):
    return orientation != "hidden"


def label_color(style, rng: np.random.Generator):
    if style.get("_force_white_text"):
        return "white"
    if style.get("labels_colored"):
        choices = [
            "#666666", "#1f77b4", "#d62728", "#2ca02c", "#9467bd",
            "#8c564b", "#e377c2", "#17becf", "#ff7f0e", "#bcbd22",
        ]
        return choices[int(rng.integers(0, len(choices)))]
    return "black"


# def background_color(style):
#     return "#f0f0f0" if style.get("background_non_white") else "white"

def background_color(style, rng: np.random.Generator | None = None):
    rng  = rng or np.random.default_rng()
    code = style.get("background", "white")
    if code == "light_gray":
        return BACKGROUND_LIGHT_GRAY[int(rng.integers(0, len(BACKGROUND_LIGHT_GRAY)))]
    if code == "dark_gray":
        return BACKGROUND_DARK_GRAY[int(rng.integers(0, len(BACKGROUND_DARK_GRAY)))]
    if code == "light_color":
        return BACKGROUND_LIGHT_COLOR[int(rng.integers(0, len(BACKGROUND_LIGHT_COLOR)))]
    if code == "dark":
        return BACKGROUND_DARK[int(rng.integers(0, len(BACKGROUND_DARK)))]
    return "white"

def title_font_size(style, rng: np.random.Generator):
    if style.get("title_size") == "large":
        return int(rng.integers(20, 25))
    if style.get("title_size") == "small":
        return int(rng.integers(10, 13))
    return int(rng.integers(12, 19))


def title_location_mpl(style):
    return {"center": "center", "left": "left", "right": "right"}.get(
        style.get("title_location", "center"), "center")


def title_anchor_altair(style):
    return {"center": "middle", "left": "start", "right": "end"}.get(
        style.get("title_location", "center"), "middle")


def legend_location_mpl(location):
    return {
        "left":     "center left",  "right":    "center right",
        "top":      "upper center", "bottom":   "lower center",
        "top-left": "upper left",   "top-right": "upper right",
        "no legend": None,
    }.get(location, "best")


def legend_location_altair(location):
    if location in {"no legend", None}:
        return None
    return location if location in _ALTAIR_VALID_ORIENTS else "right"


def grid_axes(style):
    if not style.get("gridlines"):
        return False, False
    if style.get("gridlines_both"):
        return True, True
    return (False, True) if style.get("is_vertical") else (True, False)


# Large pool of visually distinct colors to draw from when palettes run short
_DISTINCT_COLOR_POOL = list(dict.fromkeys([
    c for palette in PALETTES.values() for c in palette
]))

def bar_colors(style, n, rng: np.random.Generator):
    if not style.get("colored_bars", True):
        idx = int(rng.integers(0, len(COLOR_PALETTE_HEX)))
        return [COLOR_PALETTE_HEX[idx]] * n
    palette = list(PALETTES.get(style.get("palette_type", 4), PALETTES[4]))
    # If palette too small, supplement with distinct colors from other palettes
    if len(palette) < n:
        extras = [c for c in _DISTINCT_COLOR_POOL if c not in palette]
        rng.shuffle(extras)
        palette = palette + extras
    # Pick n unique colors, shuffled
    indices = list(rng.choice(len(palette), size=n, replace=False))
    return [palette[i] for i in indices]


## 4. Title generation


In [68]:
_groq_client = Groq()   # reads GROQ_API_KEY from env

MAX_TITLE_CHARS = 60


def make_title(context: dict, style: dict | None = None) -> tuple[str, str | None]:
    value_col    = context.get("value_col", "Value")
    category_col = context.get("category_col", "Category")

    filter_parts = [f"{k[7:]}: {v}" for k, v in context.items() if k.startswith("filter_")]
    filter_desc  = f" (filtered to {', '.join(filter_parts)})" if filter_parts else ""

    time_desc = ""
    if "start" in context:
        start_str = pd.to_datetime(context["start"]).strftime("%B %Y")
        end_str   = pd.to_datetime(context["end"]).strftime("%B %Y")
        time_desc = f", covering {start_str} to {end_str}"

    need_subtitle = style is not None and style.get("subtitle_present", False)

    system_prompt = (
        "You generate short, realistic chart titles for bar charts — the kind you'd see "
        "in a business report or dashboard. Keep titles under 60 characters. "
        "Be concise and natural. No quotes, no markdown."
    )
    user_prompt = (
        f"Generate a chart title for a bar chart showing {value_col} "
        f"by {category_col}{filter_desc}{time_desc}.\n"
    )
    if need_subtitle:
        user_prompt += (
            "Also generate a short subtitle (one line, adds context or time range detail). "
            "Respond in this exact format:\nTITLE: <title here>\nSUBTITLE: <subtitle here>"
        )
    else:
        user_prompt += "Respond with just the title, nothing else."

    try:
        response = _groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt},
            ],
            temperature=0.8,
            max_tokens=80,
        )
        raw = response.choices[0].message.content.strip()

        if need_subtitle and "TITLE:" in raw:
            lines = {k.strip(): v.strip() for k, v in
                     (line.split(":", 1) for line in raw.splitlines() if ":" in line)}
            return lines.get("TITLE", raw)[:MAX_TITLE_CHARS], lines.get("SUBTITLE", "")[:MAX_TITLE_CHARS]

        return raw[:MAX_TITLE_CHARS], None

    except Exception:
        fallback_title = f"{value_col} by {category_col}"[:MAX_TITLE_CHARS]
        fallback_sub   = "Business report" if need_subtitle else None
        return fallback_title, fallback_sub


## 5. Shared plotting helpers


In [69]:
def apply_mpl_common_style(fig, ax, style, title, subtitle, rng):
    bg = background_color(style, rng)
    ax.set_facecolor(bg)
    fig.patch.set_facecolor(bg)

    if style.get("title_present") and title:
        title_text = f"{title}\n{subtitle}" if subtitle else title
        ax.set_title(
            title_text,
            loc=title_location_mpl(style),
            color=style.get("title_color", "#000000"),
            fontsize=title_font_size(style, rng),
        )

    if style.get("outline_chart"):
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_color("black")
            spine.set_linewidth(1)
    else:
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_visible(True)
        ax.spines["bottom"].set_visible(True)

    grid_x, grid_y = grid_axes(style)
    if grid_x or grid_y:
        axis = "both" if (grid_x and grid_y) else ("x" if grid_x else "y")
        ax.grid(True, axis=axis, linestyle="-", alpha=0.3, color="gray")
        ax.set_axisbelow(True)
    else:
        ax.grid(False)

    lx      = label_color(style, rng)
    x_angle = axis_angle(style.get("x_labels_orientation", "horizontal"))
    y_angle = axis_angle(style.get("y_labels_orientation", "horizontal"))

    if labels_visible(style.get("x_labels_orientation", "horizontal")):
        plt.setp(ax.get_xticklabels(), rotation=x_angle, color=lx)
        ax.tick_params(axis="x", colors=lx,
                       length=3 if style.get("ticks") in ["x", "both"] else 0)
    else:
        ax.set_xticklabels([])
        ax.tick_params(axis="x", length=0)

    if labels_visible(style.get("y_labels_orientation", "horizontal")):
        plt.setp(ax.get_yticklabels(), rotation=y_angle, color=lx)
        ax.tick_params(axis="y", colors=lx,
                       length=3 if style.get("ticks") in ["y", "both"] else 0)
    else:
        ax.set_yticklabels([])
        ax.tick_params(axis="y", length=0)


def add_mpl_bar_labels(ax, plot_df, value_col, style):
    mode = style.get("direct_labeling", "none")
    if mode == "none":
        return
    if mode == "direct_no_axis":
        if style.get("is_vertical"):
            ax.set_yticklabels([])
        else:
            ax.set_xticklabels([])
    for patch, value in zip(ax.patches, plot_df[value_col]):
        label = str(int(round(value))) if abs(value - round(value)) < 1e-9 else f"{value:,.1f}"
        if style.get("is_vertical"):
            x = patch.get_x() + patch.get_width() / 2
            if style.get("direct_label_position") == "center":
                y, va = patch.get_height() / 2, "center"
            elif style.get("direct_label_position") == "outside":
                y, va = patch.get_height(), "bottom"
            else:
                y, va = max(1, patch.get_height() * 0.05), "bottom"
            ax.text(x, y, label, ha="center", va=va, fontsize=9)
        else:
            y = patch.get_y() + patch.get_height() / 2
            if style.get("direct_label_position") == "center":
                x, ha = patch.get_width() / 2, "center"
            elif style.get("direct_label_position") == "outside":
                x, ha = patch.get_width(), "left"
            else:
                x, ha = max(1, patch.get_width() * 0.05), "left"
            ax.text(x, y, label, va="center", ha=ha, fontsize=9)


def add_mpl_legend(ax, plot_df, category_col, colors, style):
    loc = legend_location_mpl(style.get("legend_location", "no legend"))
    if loc is None or not style.get("colored_bars", True):
        existing = ax.get_legend()
        if existing:
            existing.remove()
        return
    handles = [mpatches.Patch(color=c, label=l)
               for c, l in zip(colors, plot_df[category_col])]
    kwargs = {"loc": loc, "title": category_col if style.get("legend_title") else None}
    if style.get("legend_outline"):
        kwargs.update({"frameon": True, "edgecolor": "black", "framealpha": 1})
    else:
        kwargs.update({"frameon": False})
    legend = ax.legend(handles=handles, **kwargs)
    if style.get("legend_title_color") and legend.get_title():
        legend.get_title().set_color("#555555")


## 6. Bar renderers


In [70]:
def render_bar_matplotlib(plot_df, category_col, value_col, title, subtitle, style, rng):
    fig, ax  = plt.subplots(figsize=figure_size(style))
    colors   = bar_colors(style, len(plot_df), rng)
    edgecolor = "black" if style.get("outline_bars") else "none"
    linewidth =  1      if style.get("outline_bars") else 0
    if style.get("is_vertical"):
        bars = ax.bar(plot_df[category_col], plot_df[value_col],
                      color=colors, edgecolor=edgecolor, linewidth=linewidth)
    else:
        bars = ax.barh(plot_df[category_col], plot_df[value_col],
                       color=colors, edgecolor=edgecolor, linewidth=linewidth)
    if style.get("rounded_corners"):
        for patch in bars:
            patch.set_alpha(0.92)
    ax.set_xlabel("")
    ax.set_ylabel("")
    apply_mpl_common_style(fig, ax, style, title, subtitle, rng)
    add_mpl_bar_labels(ax, plot_df, value_col, style)
    add_mpl_legend(ax, plot_df, category_col, colors, style)
    fig.tight_layout()
    return fig


def render_bar_seaborn(plot_df, category_col, value_col, title, subtitle, style, rng):
    if sns is None:
        raise ImportError("seaborn is not installed")
    fig, ax   = plt.subplots(figsize=figure_size(style))
    edgecolor = "black" if style.get("outline_bars") else "none"
    linewidth =  1      if style.get("outline_bars") else 0
    plot_args = {"edgecolor": edgecolor, "linewidth": linewidth}
    if style.get("colored_bars", True):
        scheme = SCHEME_NAMES[int(rng.integers(0, len(SCHEME_NAMES)))]
        plot_args.update({"hue": category_col, "palette": MPL_SCHEMES[scheme], "dodge": False})
    else:
        idx = int(rng.integers(0, len(COLOR_PALETTE_HEX)))
        plot_args.update({"color": COLOR_PALETTE_HEX[idx]})
    if style.get("is_vertical"):
        sns.barplot(data=plot_df, x=category_col, y=value_col, ax=ax, **plot_args)
    else:
        sns.barplot(data=plot_df, x=value_col, y=category_col, orient="h", ax=ax, **plot_args)
    colors = [p.get_facecolor() for p in ax.patches]
    ax.set_xlabel("")
    ax.set_ylabel("")
    apply_mpl_common_style(fig, ax, style, title, subtitle, rng)
    add_mpl_bar_labels(ax, plot_df, value_col, style)
    if style.get("colored_bars", True):
        existing = ax.get_legend()
        if existing:
            existing.remove()
        add_mpl_legend(ax, plot_df, category_col, colors, style)
    fig.tight_layout()
    return fig


def legend_location_plotly(location):
    return {
        "left":      {"x": 0.01, "y": 0.5,   "xanchor": "left",   "yanchor": "middle", "orientation": "v"},
        "right":     {"x": 0.99, "y": 0.5,   "xanchor": "right",  "yanchor": "middle", "orientation": "v"},
        "top":       {"x": 0.5,  "y": 1.02,  "xanchor": "center", "yanchor": "bottom", "orientation": "h"},
        "bottom":    {"x": 0.5,  "y": -0.15, "xanchor": "center", "yanchor": "top",    "orientation": "h"},
        "top-left":  {"x": 0.01, "y": 1.02,  "xanchor": "left",   "yanchor": "bottom", "orientation": "h"},
        "top-right": {"x": 0.99, "y": 1.02,  "xanchor": "right",  "yanchor": "bottom", "orientation": "h"},
    }.get(location, {"x": 1.0, "y": 1.0, "xanchor": "right", "yanchor": "top", "orientation": "v"})


def color_to_plotly(color):
    try:
        return mcolors.to_hex(color)
    except Exception:
        return str(color)


def render_bar_plotly(plot_df, category_col, value_col, title, subtitle, style, rng):
    global go
    if go is None:
        go = ensure_module("plotly.graph_objects", "plotly")

    fig        = go.Figure()
    raw_colors = bar_colors(style, len(plot_df), rng)
    colors     = [color_to_plotly(c) for c in raw_colors]
    showlegend = style.get("legend_location") != "no legend" and style.get("colored_bars", True)
    text_mode  = style.get("direct_labeling", "none") != "none"
    marker_line_color = "black" if style.get("outline_bars") else "rgba(0,0,0,0)"
    marker_line_width = 1       if style.get("outline_bars") else 0
    text_pos = {"base": "inside", "center": "inside", "outside": "outside"}.get(
        style.get("direct_label_position", "base"), "inside")

    for idx, row in plot_df.reset_index(drop=True).iterrows():
        color     = colors[idx % len(colors)]
        val_label = (str(int(round(row[value_col])))
                     if abs(row[value_col] - round(row[value_col])) < 1e-9
                     else f"{row[value_col]:,.1f}")
        common = {
            "name": row[category_col],
            "marker": {"color": color,
                       "line": {"color": marker_line_color, "width": marker_line_width}},
            "showlegend": showlegend,
            "text":         [val_label] if text_mode else None,
            "textposition": text_pos    if text_mode else None,
            "cliponaxis":   False,
        }
        if style.get("is_vertical"):
            fig.add_trace(go.Bar(x=[row[category_col]], y=[row[value_col]], **common))
        else:
            fig.add_trace(go.Bar(x=[row[value_col]], y=[row[category_col]],
                                 orientation="h", **common))

    width_px, height_px = [int(v * 110) for v in figure_size(style)]
    title_text = None
    if style.get("title_present") and title:
        title_text = title if not subtitle else f"{title}<br><sup>{subtitle}</sup>"
    title_x  = {"left": 0.01, "center": 0.5, "right": 0.99}.get(
        style.get("title_location", "center"), 0.5)
    bg       = background_color(style, rng)
    grid_x, grid_y = grid_axes(style)
    tick_color = label_color(style, rng)

    legend_dict = {"traceorder": "normal"}
    if showlegend:
        legend_dict.update(legend_location_plotly(style.get("legend_location", "right")))
        if style.get("legend_title"):
            legend_dict["title"] = {"text": category_col}
            if style.get("legend_title_color"):
                legend_dict["title"]["font"] = {"color": "#555555"}
        if style.get("legend_outline"):
            legend_dict.update({"bordercolor": "black", "borderwidth": 1,
                                 "bgcolor": "rgba(255,255,255,0.9)"})

    fig.update_layout(
        width=width_px, height=height_px,
        plot_bgcolor=bg, paper_bgcolor=bg,
        title={
            "text": title_text, "x": title_x,
            "font": {"size": title_font_size(style, rng),
                     "color": "black" if style.get("title_color") == "black" else "#555555"},
        } if title_text else None,
        showlegend=showlegend,
        legend=legend_dict,
        margin={"l": 60, "r": 40, "t": 80 if title_text else 40, "b": 60},
    )
    fig.update_xaxes(
        showgrid=grid_x, gridcolor="rgba(120,120,120,0.3)",
        tickfont={"color": tick_color},
        showticklabels=labels_visible(style.get("x_labels_orientation", "horizontal")),
        tickangle=axis_angle(style.get("x_labels_orientation", "horizontal")),
        ticks="outside" if style.get("ticks") in ["x", "both"] else "",
        showline=style.get("outline_chart"), linecolor="black",
        mirror=style.get("outline_chart"),
    )
    fig.update_yaxes(
        showgrid=grid_y, gridcolor="rgba(120,120,120,0.3)",
        tickfont={"color": tick_color},
        showticklabels=labels_visible(style.get("y_labels_orientation", "horizontal")),
        tickangle=axis_angle(style.get("y_labels_orientation", "horizontal")),
        ticks="outside" if style.get("ticks") in ["y", "both"] else "",
        showline=style.get("outline_chart"), linecolor="black",
        mirror=style.get("outline_chart"),
    )
    if style.get("direct_labeling") == "direct_no_axis":
        if style.get("is_vertical"):
            fig.update_yaxes(showticklabels=False)
        else:
            fig.update_xaxes(showticklabels=False)
    return fig


def ensure_altair_available():
    global alt
    if alt is None:
        alt = ensure_module("altair", "altair")
    return alt


def ensure_altair_png_support():
    try:
        import vl_convert  # noqa: F401
    except Exception:
        install_package("vl-convert-python")


def render_bar_altair(plot_df, category_col, value_col, title, subtitle, style, rng):
    ensure_altair_available()
    grid_x, grid_y = grid_axes(style)
    lx     = label_color(style, rng)
    scheme = SCHEME_NAMES[int(rng.integers(0, len(SCHEME_NAMES)))]

    x_axis = alt.Axis(
        labels=labels_visible(style.get("x_labels_orientation", "horizontal")),
        labelAngle=axis_angle(style.get("x_labels_orientation", "horizontal"), backend="altair"),
        labelColor=lx, title=None, grid=grid_x,
        ticks=style.get("ticks") in ["x", "both"],
    )
    y_axis = alt.Axis(
        labels=labels_visible(style.get("y_labels_orientation", "horizontal")),
        labelAngle=axis_angle(style.get("y_labels_orientation", "horizontal"), backend="altair"),
        labelColor=lx, title=None, grid=grid_y,
        ticks=style.get("ticks") in ["y", "both"],
    )

    if style.get("colored_bars", True):
        legend_loc = legend_location_altair(style.get("legend_location", "no legend"))
        legend = (alt.Legend(orient=legend_loc,
                             title=category_col if style.get("legend_title") else None)
                  if legend_loc else None)
        color_encode = alt.Color(f"{category_col}:N",
                                 scale=alt.Scale(scheme=scheme), legend=legend)
    else:
        idx = int(rng.integers(0, len(COLOR_PALETTE_HEX)))
        color_encode = alt.value(COLOR_PALETTE_HEX[idx])

    mark_args = {
        "cornerRadiusEnd": 4 if style.get("rounded_corners") else 0,
        "stroke":          "black" if style.get("outline_bars") else "transparent",
        "strokeWidth":     1       if style.get("outline_bars") else 0,
    }

    if style.get("is_vertical"):
        chart = alt.Chart(plot_df).mark_bar(**mark_args).encode(
            x=alt.X(f"{category_col}:N", axis=x_axis),
            y=alt.Y(f"{value_col}:Q",    axis=y_axis),
            color=color_encode,
        )
    else:
        chart = alt.Chart(plot_df).mark_bar(**mark_args).encode(
            x=alt.X(f"{value_col}:Q",    axis=x_axis),
            y=alt.Y(f"{category_col}:N", axis=y_axis),
            color=color_encode,
        )

    if style.get("direct_labeling", "none") != "none":
        if style.get("is_vertical"):
            text_mark = alt.Chart(plot_df).mark_text(
                dy=-6 if style.get("direct_label_position") == "outside" else 0
            ).encode(x=f"{category_col}:N", y=f"{value_col}:Q", text=f"{value_col}:Q")
        else:
            text_mark = alt.Chart(plot_df).mark_text(
                dx=8    if style.get("direct_label_position") == "outside" else 0,
                align="left" if style.get("direct_label_position") == "outside" else "center",
            ).encode(x=f"{value_col}:Q", y=f"{category_col}:N", text=f"{value_col}:Q")
        chart = chart + text_mark

    props = {
        "width":  int(figure_size(style)[0] * 80),
        "height": int(figure_size(style)[1] * 80),
    }
    if style.get("title_present") and title:
        title_kwargs = {
            "text":     title,
            "anchor":   title_anchor_altair(style),
            "color":    "black" if style.get("title_color") == "black" else "#555555",
            "fontSize": title_font_size(style, rng),
        }
        if subtitle:
            title_kwargs["subtitle"] = subtitle
        props["title"] = alt.TitleParams(**title_kwargs)

    chart = chart.properties(**props).configure_view(
        stroke="black" if style.get("outline_chart") else "transparent",
        strokeWidth=1,
        fill=background_color(style, rng),
    )
    if style.get("direct_labeling") == "direct_no_axis":
        if style.get("is_vertical"):
            chart = chart.configure_axisY(labels=False)
        else:
            chart = chart.configure_axisX(labels=False)
    return chart


## 7. Saving and generation


In [71]:
def ensure_output_dirs(out_root: Path) -> None:
    if CLEAR_OUTPUT and out_root.exists():
        shutil.rmtree(out_root)
    for sub in SUBDIRS:
        for lib in LIBRARIES:
            (out_root / sub / lib).mkdir(parents=True, exist_ok=True)


def chromium_executable():
    for name in ["chromium", "chromium-browser", "google-chrome", "google-chrome-stable"]:
        found = shutil.which(name)
        if found:
            return found
    return None


def save_html_screenshot(html_content: str, path: Path, width: int = 1000, height: int = 700):
    path = Path(path).with_suffix(".png")
    path.parent.mkdir(parents=True, exist_ok=True)
    chrome = chromium_executable()
    if chrome is None:
        raise RuntimeError("Chromium/Chrome required for HTML-to-PNG fallback.")
    with tempfile.TemporaryDirectory() as tmpdir:
        html_path = Path(tmpdir) / "chart.html"
        html_path.write_text(html_content, encoding="utf-8")
        subprocess.check_call([
            chrome, "--headless", "--disable-gpu", "--no-sandbox",
            f"--window-size={int(width)},{int(height)}",
            "--hide-scrollbars", "--force-device-scale-factor=1",
            "--virtual-time-budget=2000",
            f"--screenshot={str(path)}", html_path.as_uri(),
        ])
    return path


def save_matplotlib_png(fig, path: Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return path


def save_plotly(fig, path: Path) -> Path:
    path = Path(path).with_suffix(".png")
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        fig.write_image(str(path))
        return path
    except Exception:
        try:
            install_package("kaleido")
            fig.write_image(str(path))
            return path
        except Exception:
            html   = fig.to_html(include_plotlyjs="cdn", full_html=True)
            width  = int(fig.layout.width  or 1000)
            height = int(fig.layout.height or 700)
            return save_html_screenshot(html, path, width=width, height=height)


def save_altair(chart, path: Path) -> Path:
    path = Path(path).with_suffix(".png")
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        ensure_altair_png_support()
        chart.save(str(path))
        return path
    except Exception:
        try:
            html   = chart.to_html()
            width  = int(getattr(chart, "width",  800) or 800) + 120
            height = int(getattr(chart, "height", 500) or 500) + 120
            return save_html_screenshot(html, path, width=width, height=height)
        except Exception as exc:
            raise RuntimeError("Altair PNG export failed.") from exc


def save_metadata(meta: dict, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2, default=str)


def generate_bar(
    out_root: Path,
    dataset_source: str,
    library: str,
    rng_seed: int,
    weights: dict,
    datasets: dict | None = None,
    max_tries: int = 50,
) -> dict:
    rng      = np.random.default_rng(rng_seed)
    chart_id = new_chart_id("bar")

    for attempt in range(1, max_tries + 1):
        style = harmonize_style(sample_style(rng, weights))

        plot_df = None
        context = None

        if datasets:
            ds_names = list(datasets.keys())
            ds_name  = ds_names[int(rng.integers(0, len(ds_names)))]
            df       = datasets[ds_name]
            spec     = DATASET_REGISTRY[ds_name]
            result   = sample_bar_data_from_df(df, spec, rng, style)
            if result is not None:
                plot_df, context = result
                dataset_source   = ds_name
            else:
                print(f"[seed {rng_seed}] Real data sampling failed on attempt {attempt}")

        if plot_df is None:
            continue

        category_col    = context["category_col"]
        value_col       = context["value_col"]
        title, subtitle = make_title(context, style=style)

        table_path = out_root / "tables"   / library / f"{chart_id}.csv"
        meta_path  = out_root / "metadata" / library / f"{chart_id}.json"
        table_path.parent.mkdir(parents=True, exist_ok=True)
        plot_df.to_csv(table_path, index=False)

        if library == "altair":
            image_path = out_root / "images" / library / f"{chart_id}.png"
            chart = render_bar_altair(plot_df, category_col, value_col, title, subtitle, style, rng)
            save_altair(chart, image_path)
        elif library == "matplotlib":
            image_path = out_root / "images" / library / f"{chart_id}.png"
            fig = render_bar_matplotlib(plot_df, category_col, value_col, title, subtitle, style, rng)
            save_matplotlib_png(fig, image_path)
        elif library == "seaborn":
            image_path = out_root / "images" / library / f"{chart_id}.png"
            fig = render_bar_seaborn(plot_df, category_col, value_col, title, subtitle, style, rng)
            save_matplotlib_png(fig, image_path)
        elif library == "plotly":
            image_path = out_root / "images" / library / f"{chart_id}.png"
            fig = render_bar_plotly(plot_df, category_col, value_col, title, subtitle, style, rng)
            save_plotly(fig, image_path)
        else:
            raise ValueError(f"Unsupported library: {library}")

        meta = {
            "chart_id":       chart_id,
            "chart_type":     "bar",
            "library":        library,
            "dataset_source": dataset_source,
            "image_path":     str(image_path),
            "table_path":     str(table_path),
            "data_context":   context,
            "style":          style,
            "created_utc":    datetime.utcnow().isoformat() + "Z",
            "rng_seed":       rng_seed,
        }
        save_metadata(meta, meta_path)
        return meta

    raise RuntimeError(f"Failed to generate bar after {max_tries} attempts.")


def generate_batch(
    out_root: Path,
    dataset_source: str,
    generation_plan: dict,
    weights: dict,
    datasets: dict | None = None,
    start_seed: int = 1000,
) -> list[dict]:
    ensure_output_dirs(out_root)
    metas = []
    seed  = start_seed
    for library, n in generation_plan.items():
        for _ in range(n):
            metas.append(generate_bar(
                out_root=out_root,
                dataset_source=dataset_source,
                library=library,
                rng_seed=seed,
                weights=weights,
                datasets=datasets,
            ))
            seed += 1
    return metas


## 8. Batch generation


In [72]:
ensure_output_dirs(BAR_OUT_ROOT)

weights = normalize_weights(SAMPLING_WEIGHTS)

generation_plan = {
    "altair":     10,
    "matplotlib": 10,
    "seaborn":    10,
    "plotly":     10,
}

metas = generate_batch(
    out_root=BAR_OUT_ROOT,
    dataset_source="warehouse_retail",
    generation_plan=generation_plan,
    weights=weights,
    datasets=DATASETS,
    start_seed=666,
)

pd.DataFrame(metas)[["chart_id", "library", "dataset_source", "image_path"]].head()


,chart_id,library,dataset_source,image_path
0,bar_20260609T191522_d09b01bd,altair,warehouse_retail,C:\Users\Michelle\I2R\notebooks\final_notebook...
1,bar_20260609T191523_29c317c3,altair,warehouse_retail,C:\Users\Michelle\I2R\notebooks\final_notebook...
2,bar_20260609T191524_e448dec1,altair,warehouse_retail,C:\Users\Michelle\I2R\notebooks\final_notebook...
3,bar_20260609T191525_7254ae4c,altair,warehouse_retail,C:\Users\Michelle\I2R\notebooks\final_notebook...
4,bar_20260609T191526_a8457c9f,altair,warehouse_retail,C:\Users\Michelle\I2R\notebooks\final_notebook...


# TESTING

### Smoke test


In [ ]:
# TEST_LIBRARIES = ["matplotlib", "seaborn", "altair", "plotly"]

# weights = normalize_weights(SAMPLING_WEIGHTS)

# # Base style: most frequent value for each parameter
# base = {param: max(probs, key=probs.get) for param, probs in weights.items()}
# base.update({
#     "title_present": True, "title_location": "center", "title_color": "black",
#     "title_size": "average", "legend_location": "right",
#     "direct_labeling": "none", "background_non_white": False,
# })

# errors = []
# rng = np.random.default_rng(42)

# for param_name, probs in weights.items():
#     for val in probs.keys():
#         style   = harmonize_style({**base, param_name: val})
#         ds_name = list(DATASETS.keys())[0]
#         df      = DATASETS[ds_name]
#         spec_ds = DATASET_REGISTRY[ds_name]
#         result  = sample_bar_data_from_df(df, spec_ds, rng, style)
#         if result is None:
#             continue
#         plot_df, context = result
#         category_col = context["category_col"]
#         value_col    = context["value_col"]
#         title, subtitle = "Test", None

#         for library in TEST_LIBRARIES:
#             try:
#                 if library == "matplotlib":
#                     fig = render_bar_matplotlib(plot_df, category_col, value_col, title, subtitle, style, rng)
#                     plt.close(fig)
#                 elif library == "seaborn":
#                     fig = render_bar_seaborn(plot_df, category_col, value_col, title, subtitle, style, rng)
#                     plt.close(fig)
#                 elif library == "altair":
#                     render_bar_altair(plot_df, category_col, value_col, title, subtitle, style, rng)
#                 elif library == "plotly":
#                     render_bar_plotly(plot_df, category_col, value_col, title, subtitle, style, rng)
#             except Exception as e:
#                 errors.append({"param": param_name, "value": val, "library": library,
#                                "error": f"{type(e).__name__}: {e}"})

# if errors:
#     print(f"Found {len(errors)} errors:")
#     pd.DataFrame(errors)
# else:
#     print("All parameter values passed for all libraries.")


All parameter values passed for all libraries.
